In [1]:
import robotic as ry
import numpy as np
import time
import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
"""
Proof-of-concept: spawn N blocks at Gaussian-sampled heights and let them fall


This script creates a table, spawns blocks with Z sampled from a normal
distribution, and runs the physx simulation.
"""


def spawn_blocks(C, n_blocks=10, mean_z=1.0, std_z=0.1,
                 x_range=(-0.4, 0.4), y_range=(-0.4, 0.4),
                 block_size=(0.06, 0.06, 0.06, 0.01), seed=None):
    """Add `n_blocks` to Config C. Z heights are sampled from N(mean_z, std_z).

    Arguments:
        C: ry.Config()
        n_blocks: how many blocks to spawn
        mean_z, std_z: gaussian parameters (meters)
        x_range, y_range: uniform sampling range for X/Y
        block_size: shape size for ry.ST.ssBox (lengths + roundness)
        seed: RNG seed for reproducibility
    Returns:
        list of block frame names
    """
    if seed is not None:
        np.random.seed(seed)

    # table height
    table = C.getFrame('table')
    table_z = table.getPosition()[2]
    half_height = block_size[2] / 2.0
    min_z = table_z + half_height + 0.01

    names = []
    zs = np.random.normal(loc=mean_z, scale=std_z, size=n_blocks)
    for i in range(n_blocks):
        name = f'block_{i}'
        x = float(np.random.uniform(*x_range))
        y = float(np.random.uniform(*y_range))
        z = float(max(zs[i], min_z))  # avoid spawning inside the table

        f = C.addFrame(name)
        f.setShape(ry.ST.ssBox, size=list(block_size))
        f.setPosition([x, y, z])
        f.setColor([0.7, 0.3 + 0.02*i, 0.2 + 0.05*i]) # SIMPLE RANDOM SAMPLING FOR COLORS CAN BE IMPROVED LATER
        f.setMass(0.2)
        f.setContact(True)
        # MAYBE SET FRICTION AND OTHER PARAMS LATTER
        names.append(name)

    return names

In [ ]:
C = ry.Config()
C.addFile("my_environment.g")
C.view(False, "waitfor close")

0

-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled


In [4]:
# parameters (tweak as you like)
N_BLOCKS = 1
MEAN_Z = 1.0
STD_Z = 0.15
SIM_DT = 0.01
SIM_SECONDS = 1.0



# Table
C.addFrame('table') \
    .setShape(ry.ST.ssBox, size=[1.2, 1.0, 0.08, 0.02]) \
    .setColor([0.3, 0.3, 0.35]) \
    .setPosition([0, 0, 0.4])

# Blocks
spawn_blocks(C, n_blocks=N_BLOCKS, mean_z=MEAN_Z, std_z=STD_Z, seed=42)

C.view(False)

S = ry.Simulation(C, ry.SimulationEngine.physx, verbose=0)

steps = int(np.ceil(SIM_SECONDS / SIM_DT))
for i in range(steps):
    S.step([], SIM_DT, ry.ControlMode.none)

    if i % 2 == 0:
        C.view()
    time.sleep(SIM_DT)

C.view(False, "waitfor close")
# del S


0

In [6]:
# C.watchFile("my_environment.g")


In [5]:
cameras = ["camera1", "camera2", "camera3", "camera4"]

In [6]:

rgb_colors = [[255,0,0], [0,255,0], [0,0,255], [255,255,0]]
pcl_all = []
for i, camera in enumerate(cameras):
    cam = ry.CameraView(C)
    cam.setCamera(C.getFrame(camera))
    R_WC = C.getFrame(camera).getRotationMatrix()

    rgb, depth = cam.computeImageAndDepth(C)

    # plt.figure(figsize=(10,10))
    # plt.title(f"Camera {camera}")
    # plt.subplot(1, 2 ,1)
    # plt.imshow(rgb)

    # plt.subplot(1, 2 ,2)
    # plt.imshow(depth)
    # plt.show()

    pcl = ry.depthImage2PointCloud(depth, cam.getFxycxy())
    pcl = pcl.reshape(-1, 3)

    # f = C.addFrame(f"pcl{i}", camera)
    # f.setPointCloud(pcl, rgb_colors[i])

    for point in pcl:
        p_WP = R_WC @ point + C.getFrame(camera).getPosition()
        pcl_all.append(p_WP)
# C.view(True)

-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled
-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled
-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled
-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled


In [7]:
bbox_min_pos = C.getFrame("bbox_min").getPosition()
bbox_max_pos = C.getFrame("bbox_max").getPosition()

x_min, x_max = bbox_min_pos[0], bbox_max_pos[0]
y_min, y_max = bbox_min_pos[1], bbox_max_pos[1]
z_min, z_max = bbox_min_pos[2], bbox_max_pos[2]

# z_min, z_max = 0.7121, 1.04

cropped_pcl = []
for p in pcl_all:
    x, y, z = p

    if x_min < x < x_max and y_min < y < y_max and z_min < z < z_max:
        cropped_pcl.append(p)

assert len(cropped_pcl) > 0, "No points in the cropped area"
cropped_pcl = np.array(cropped_pcl)

f = C.addFrame("cropped_pcl", "world")
f.setPointCloud( cropped_pcl, [0, 0, 255])
C.view(False, "cropped point cloud, # of points: " + str(len(cropped_pcl)))
print(f"Total number of points: {len(cropped_pcl)}")

Total number of points: 2950


In [10]:
# C.view(True, "waitfor close")

In [8]:
# import open3d as o3d
import numpy as np

def align_normals_with_centroid(point_cloud):
    """
    Aligns normals based on the global centroid of the point cloud.
    """
    normals = np.asarray(point_cloud.normals)
    points = np.asarray(point_cloud.points)

    # Global Centroid-Based Orientation
    centroid = np.mean(points, axis=0)
    for i in range(len(normals)):
        to_point = points[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    """
    Aligns normals based on the average normal of local neighbors.
    """
    normals = np.asarray(point_cloud.normals)
    points = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        # Find the 10 nearest neighbors
        [_, idx, _] = kdtree.search_knn_vector_3d(points[i], 10)
        # Calculate the average normal in this neighborhood
        avg_normal = np.mean(normals[idx], axis=0)
        avg_normal /= np.linalg.norm(avg_normal)
        if np.dot(normals[i], avg_normal) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud




point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)

point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))

# Align normals based on centroid
aligned_point_cloud = align_normals_with_centroid(point_cloud)

# Reorient normals locally after bounding box flipping
reoriented_point_cloud = reorient_normals_locally(aligned_point_cloud)


normals = np.asarray(reoriented_point_cloud.normals)
print(f"Number of processed_point_cloud points: {len(reoriented_point_cloud.points)}")
o3d.visualization.draw_geometries([reoriented_point_cloud], point_show_normal=True)


Number of processed_point_cloud points: 2950


In [9]:
def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

min_distance = 0.01
max_distance = 0.09
small_angle_threshold = 10  # Max angle 
large_angle_threshold = 160  # Min angle 

best_grasps = []
compute_grasp_points = True
if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []

    centroid = np.mean(points, axis=0)

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)
            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)
                if 160 <= angle <= 180:
                    # Compute the epipolar line (direction vector)
                    epiline = p2 - p1
                    epiline /= np.linalg.norm(epiline)

                    # Check normal directions relative to the epipolar line
                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                  
                    if projection_n1 < 0 and projection_n2 > 0:
                        # Calculate the angles between the epipolar line and each normal
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)
                        if min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold:
                            # Calculate closeness to the centroid
                            dist_to_centroid = np.linalg.norm((p1 + p2) / 2 - centroid)
                            grasp_candidates.append((p1, p2, n1, n2, angle, distance, angle_n1_epiline, angle_n2_epiline, dist_to_centroid))
    if grasp_candidates:
        # Select the best grasp considering  closeness to the centroid
        best_grasp = min(grasp_candidates, key=lambda x: x[8] * 1e1)  # Minimize closeness to centroid
        best_grasps.append(best_grasp)
        print(f"Best grasp candidate:", best_grasp)
    else:
        print(f"No suitable grasp candidates found")
        grasp_candidates.append(None)


Best grasp candidate: (array([0.18819947, 0.05071719, 0.48441237]), array([0.18344498, 0.10824631, 0.48699687]), array([-0.09275454, -0.99539511, -0.02419014]), array([0.06355203, 0.99794029, 0.00873603]), 178.10127435042662, 0.05778307588782285, 169.88923843411789, 8.61552676702628, 0.0002454678356970015)


In [10]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    # grasp_candidates_np = np.array(grasp_candidates, dtype=object)

    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
    # np.savez("grasp_candidates.npz", grasp_candidates=grasp_candidates_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']
    # grasp_candidates = loaded_data['grasp_candidates']

In [11]:
for i, best_grasp in enumerate(best_grasps):
    # put a marker at best grasp points
    f = C.addFrame(f"best_grasp_{i}_pt1", "world")
    f.setPosition(best_grasp[0])
    f.setColor([255, 0, 0])
    f.setShape(ry.ST.marker, [.05])
    f.setContact(0)

    # put a marker at best grasp points
    f = C.addFrame(f"best_grasp_{i}_pt2", "world")
    f.setPosition(best_grasp[1])
    f.setShape(ry.ST.marker, [.05])
    f.setColor([0, 255, 0])
    f.setContact(0)

    p1 = best_grasp[0]
    p2 = best_grasp[1]
    n1 = best_grasp[2]
    n2 = best_grasp[3]

    # diplay selected points and their normals in open3d
    point_cloud = o3d.geometry.PointCloud()

    point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
    point_cloud.colors = o3d.utility.Vector3dVector([[255, 0, 0], [0, 255, 0]])

    normals = np.array([n1, n2])
    point_cloud.normals = o3d.utility.Vector3dVector(normals)

    grasp_visualizations = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere1.translate(p1)
    sphere1.paint_uniform_color([1, 0, 0])  

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere2.translate(p2)
    sphere2.paint_uniform_color([0, 1, 0]) 

    # Add spheres to the visualization
    grasp_visualizations.append(sphere1)
    grasp_visualizations.append(sphere2)

    # Create lines to represent normals at each grasp point
    normal_line1 = o3d.geometry.LineSet()
    normal_line2 = o3d.geometry.LineSet()

    # Normal vectors n1 and n2 at points p1 and p2 respectively

    line_points1 = [p1, p1 + 0.05 * n1]  # Line representing normal direction for p1
    line_points2 = [p2, p2 + 0.05 * n2]  # Line representing normal direction for p2
   
    normal_line1.points = o3d.utility.Vector3dVector(line_points1)
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p1 to p1 + normal
    normal_line1.colors = o3d.utility.Vector3dVector([[1, 0, 0]])  # Color red

    normal_line2.points = o3d.utility.Vector3dVector(line_points2)
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p2 to p2 + normal
    normal_line2.colors = o3d.utility.Vector3dVector([[0, 1, 0]])  # Color green

    # Add normal lines to the visualization
    grasp_visualizations.append(normal_line1)
    grasp_visualizations.append(normal_line2)

    o3d.visualization.draw_geometries([point_cloud] + grasp_visualizations)
C.view(False, "Best grasp points")


0